In [3]:
# === Import Libraries dan Konfigurasi ===
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

# --- Konfigurasi ---
HONDA_DEALERS_URL = "https://www.honda-indonesia.com/dealers/"
OUTPUT_EXCEL = "honda_dealers_indonesia.xlsx"
SAVE_EVERY = 10  # simpan berkala tiap N baris
SEARCH_DELAY_SEC = 3


In [4]:
# === Step 1: Fungsi BS4 untuk Scrape Provinsi dan Dealer ===
def scrape_dealers_bs4() -> pd.DataFrame:
    """
    Scrape semua nama dealer dan provinsinya dari website Honda menggunakan BeautifulSoup.
    Hanya mengambil data Provinsi dan Dealer saja.
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    resp = requests.get(HONDA_DEALERS_URL, headers=headers, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    province_content_divs = soup.find_all('div', attrs={'x-show': lambda x: x and 'open ===' in x})
    rows = []

    for content_div in province_content_divs:
        try:
            # Ambil nama provinsi dari button sebelumnya
            button = content_div.find_previous('button')
            span = button.find('span', class_='tw-font-bold') if button else None
            province_name = span.get_text(strip=True) if span else "Unknown"

            # Ambil semua dealer di provinsi tersebut
            dealer_blocks = content_div.find_all('div', class_='tw-space-y-4')
            for block in dealer_blocks:
                name_tag = block.find('a', href=lambda x: x and '/dealers/' in x)
                dealer = name_tag.get_text(strip=True) if name_tag else "N/A"

                # Hanya simpan Provinsi dan Dealer
                rows.append({
                    'Provinsi': province_name,
                    'Dealer': dealer,
                })
        except Exception as e:
            print(f"Error saat scraping: {e}")
            continue

    df = pd.DataFrame(rows)
    # Hapus duplikat jika ada
    df = df.drop_duplicates(subset=['Provinsi', 'Dealer']).reset_index(drop=True)
    return df


In [5]:
# === Step 2: Jalankan Scraping BS4 - Ambil Semua Dealer dan Provinsi ===
print("Scraping daftar dealer dan provinsi menggunakan BS4...")
df = scrape_dealers_bs4()
print(f"Ditemukan {len(df)} dealer dari {df['Provinsi'].nunique()} provinsi")
print("\nPreview data:")
print(df.head(10))

# Siapkan kolom untuk data yang akan diisi dari Google Maps
df['Alamat'] = ''
df['Latitude'] = ''
df['Longitude'] = ''
df['Alamat Map'] = ''
df['Kabupaten/Kota'] = ''

print(f"\nTotal baris yang akan diproses: {len(df)}")


Scraping daftar dealer dan provinsi menggunakan BS4...
Ditemukan 192 dealer dari 32 provinsi

Preview data:
  Provinsi                  Dealer
0     Bali   Honda Bintang Tabanan
1     Bali      Honda Cokroaminoto
2     Bali    Honda Denpasar Agung
3     Bali      Honda Dewata Motor
4   Banten       Honda Arta Cikupa
5   Banten      Honda Auto Cilegon
6   Banten       Honda Auto Serang
7   Banten  Honda Autoland Ciputat
8   Banten    Honda Bintang Cimone
9   Banten           Honda Bintaro

Total baris yang akan diproses: 192


In [6]:
# === Step 3: Setup Selenium dan Fungsi Helper untuk Google Maps ===

# --- Setup WebDriver ---
def init_driver():
    """Inisialisasi Chrome WebDriver dengan konfigurasi anti-detection"""
    options = Options()
    options.add_argument('--start-maximized')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    return webdriver.Chrome(options=options)

# --- Fungsi Helper: Extract Koordinat dari URL ---
coord_patterns = [
    r"@(-?\d+\.?\d*),(-?\d+\.?\d*)",
    r"[?&]q=(-?\d+\.?\d*),(-?\d+\.?\d*)",
    r"/dir/(-?\d+\.?\d*),(-?\d+\.?\d*)",
    r"center=(-?\d+\.?\d*),(-?\d+\.?\d*)",
]

def extract_coordinates_from_url(url: str):
    """Extract latitude dan longitude dari URL Google Maps"""
    for pattern in coord_patterns:
        m = re.search(pattern, url)
        if m:
            lat, lon = m.group(1), m.group(2)
            try:
                if -90 <= float(lat) <= 90 and -180 <= float(lon) <= 180:
                    return lat, lon
            except:
                pass
    return "", ""

# --- Fungsi Helper: Parse Kabupaten/Kota dari Alamat ---
def parse_kabkota_from_address(alamat_map: str) -> str:
    """Extract kabupaten/kota dari alamat lengkap"""
    if not alamat_map:
        return ""
    cleaned = alamat_map.strip()
    first_space = cleaned.find(' ')
    if '+' in cleaned.split(' ')[0] and first_space != -1:
        cleaned = cleaned[first_space+1:].strip()
    parts = [p.strip() for p in cleaned.split(',') if p.strip()]
    for part in parts:
        low = part.lower()
        if low.startswith('kota ') or low.startswith('kabupaten '):
            return part
    for part in reversed(parts[-3:]):
        if (5 < len(part) < 40 and 'indonesia' not in part.lower() and not part.replace(' ', '').isdigit()):
            return part
    return ""

# --- Selector untuk mencari alamat di Google Maps ---
MAP_SELECTORS = [
    "div.MngOvd span.DkEaL",
    "button[data-section-id='194'] .DkEaL",
    "span.DkEaL",
    "div.LCF4w span.DkEaL",
    "button[data-item-id='address']",
    "[data-value='Address']",
    ".Io6YTe.fontBodyMedium",
    "button[jsaction*='address']",
    "[class*='Io6YTe']",
    "[class*='fontBodyMedium']",
    "div.qbxhc",
    "span.LrzXr",
]

# --- Fungsi Utama: Scrape Info dari Google Maps ---
def scrape_maps_info_by_name(dealer_name: str, provinsi: str, driver) -> tuple[str, str, str, str]:
    """
    Mencari informasi dealer di Google Maps berdasarkan nama dealer dan provinsi.
    Returns: (latitude, longitude, alamat_map, kabupaten_kota)
    """
    if not dealer_name or dealer_name == 'N/A':
        return "", "", "", ""
    
    # Buat query pencarian
    query = f"{dealer_name} {provinsi} Indonesia"
    maps_url = f"https://www.google.com/maps/search/{query.replace(' ', '+')}"
    
    # Buka Google Maps
    driver.get(maps_url)
    time.sleep(4)
    
    # Klik hasil pertama jika tersedia
    try:
        first_result = WebDriverWait(driver, 8).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 
                "div[role='article']:first-of-type, a[data-value='Directions'], [jsaction*='mouseover']:first-of-type"))
        )
        first_result.click()
        time.sleep(4)
    except Exception:
        pass
    
    # Extract koordinat dari URL
    current_url = driver.current_url
    lat, lon = extract_coordinates_from_url(current_url)
    
    # Extract alamat lengkap
    alamat_map = ""
    for selector in MAP_SELECTORS:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            for elem in elements:
                text = elem.text.strip()
                if text and len(text) > 15 and (',' in text or 'Jl' in text or 'Jalan' in text or '+' in text):
                    alamat_map = text
                    break
            if alamat_map:
                break
        except Exception:
            continue
    
    # Extract kabupaten/kota dari alamat
    kabkota = parse_kabkota_from_address(alamat_map)
    
    return (lat or ""), (lon or ""), alamat_map or "", kabkota or ""

print("Fungsi helper dan Selenium siap digunakan!")


Fungsi helper dan Selenium siap digunakan!


In [ ]:
# === Step 4: Loop untuk Mencari Alamat dan Koordinat di Google Maps ===

# Inisialisasi WebDriver
print("Menyalakan browser untuk Google Maps...")
driver = init_driver()

try:
    print(f"\nMemulai pencarian di Google Maps untuk {len(df)} dealer...")
    print("=" * 60)
    
    success = 0
    fail = 0
    
    for i, row in df.iterrows():
        dealer_name = str(row['Dealer']).strip()
        provinsi = str(row['Provinsi']).strip()
        
        print(f"[{i+1}/{len(df)}] {dealer_name} - {provinsi}")
        
        # Cari informasi di Google Maps
        lat, lon, alamat_map, kabkota = scrape_maps_info_by_name(dealer_name, provinsi, driver)
        
        # Update dataframe jika ada data yang ditemukan
        updated = False
        if lat:
            df.at[i, 'Latitude'] = lat
            updated = True
        if lon:
            df.at[i, 'Longitude'] = lon
            updated = True
        if alamat_map:
            df.at[i, 'Alamat Map'] = alamat_map
            updated = True
        if kabkota:
            df.at[i, 'Kabupaten/Kota'] = kabkota
            updated = True
        
        if updated:
            success += 1
            print(f"  ✓ Berhasil: lat={lat}, lon={lon}")
            if alamat_map:
                print(f"    Alamat: {alamat_map[:70]}...")
        else:
            fail += 1
            print(f"  ✗ Tidak ditemukan")
        
        # Simpan berkala
        if (i+1) % SAVE_EVERY == 0:
            df.to_excel(OUTPUT_EXCEL, index=False)
            print(f"  💾 Disimpan sementara ({i+1}/{len(df)})")
        
        # Delay antar pencarian
        time.sleep(SEARCH_DELAY_SEC)
    
    print("\n" + "=" * 60)
    print(f"Selesai! Sukses: {success} | Gagal: {fail}")
    
except KeyboardInterrupt:
    print("\n\n⚠️ Proses dihentikan oleh user. Menyimpan data yang sudah terkumpul...")
    try:
        df.to_excel(OUTPUT_EXCEL, index=False)
        print(f"✅ Data berhasil disimpan ke: {OUTPUT_EXCEL}")
    except PermissionError:
        print(f"⚠️ File Excel sedang digunakan. Mencoba menyimpan backup...")
        import datetime
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_name = OUTPUT_EXCEL.replace('.xlsx', f'_interrupted_{timestamp}.xlsx')
        try:
            df.to_excel(backup_name, index=False)
            print(f"✅ Data disimpan sebagai backup: {backup_name}")
        except:
            print(f"❌ Gagal menyimpan. Data masih di memory (variabel 'df').")
    except Exception as e:
        print(f"⚠️ Error saat menyimpan: {e}")
    
finally:
    # Tutup browser
    try:
        driver.quit()
        print("Browser ditutup.")
    except:
        pass


Menyalakan browser untuk Google Maps...

Memulai pencarian di Google Maps untuk 192 dealer...
[1/192] Honda Bintang Tabanan - Bali
  ✓ Berhasil: lat=-8.5530323, lon=115.1362066
    Alamat: 
Jl. Dr. Ir. Soekarno No.15, Banjar Anyar, Kec. Kediri, Kabupaten Tab...
[2/192] Honda Cokroaminoto - Bali
  ✓ Berhasil: lat=-8.6412412, lon=115.2092679
    Alamat: 
Jl. Cokroaminoto No.62, Pemecutan Kaja, Kec. Denpasar Utara, Kota De...
[3/192] Honda Denpasar Agung - Bali
  ✓ Berhasil: lat=-8.6573909, lon=115.2269249
    Alamat: 
Jl. Hayam Wuruk No.40, Sumerta Kauh, Kec. Denpasar Tim., Kota Denpas...
[4/192] Honda Dewata Motor - Bali
  ✓ Berhasil: lat=-8.6677953, lon=115.2057992
[5/192] Honda Arta Cikupa - Banten
  ✓ Berhasil: lat=-6.2229553, lon=106.5289686
[6/192] Honda Auto Cilegon - Banten
  ✓ Berhasil: lat=-6.0486576, lon=106.0603991
    Alamat: HONDA AUTO CILEGON
4,8(207)
Dealer Honda ·  · Jl. Lingkar Selatan No....
[7/192] Honda Auto Serang - Banten
  ✓ Berhasil: lat=-6.2189176, lon=106.1

PermissionError: [Errno 13] Permission denied: 'honda_dealers_indonesia.xlsx'

In [ ]:
# === Step 5: Simpan Hasil Akhir ke Excel ===

# Reorder kolom sesuai urutan yang diinginkan
final_cols = ['Provinsi', 'Dealer', 'Alamat', 'Latitude', 'Longitude', 'Alamat Map', 'Kabupaten/Kota']
df = df[final_cols]

# Simpan ke Excel
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"✅ Data berhasil disimpan ke: {OUTPUT_EXCEL}")

# Tampilkan preview
print("\n" + "=" * 60)
print("PREVIEW DATA HASIL SCRAPING:")
print("=" * 60)
print(df.head(10))

print("\n" + "=" * 60)
print("STATISTIK:")
print(f"Total dealer: {len(df)}")
print(f"Dealer dengan koordinat: {len(df[(df['Latitude'] != '') & (df['Longitude'] != '')])}")
print(f"Dealer dengan alamat map: {len(df[df['Alamat Map'] != ''])}")
print(f"Dealer dengan kabupaten/kota: {len(df[df['Kabupaten/Kota'] != ''])}")
print("=" * 60)